# ARIA v2.0 Terrain Risk Audit

Week 4 homework notebook: combine **river distance + DEM elevation + slope** to audit shelter terrain risk in **Hualien County**.

## Captain's Log 1 — Environment and thresholds

Load packages, optional `.env` thresholds, and define local paths. If `.env` is missing, the notebook falls back to sensible defaults.

In [ ]:

import geopandas as gpd
import pandas as pd
import numpy as np
import rioxarray as rxr
from rasterstats import zonal_stats
import matplotlib.pyplot as plt
from shapely.geometry import Point
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

import os
import warnings
warnings.filterwarnings("ignore")

# Optional .env loading
if load_dotenv is not None:
    load_dotenv()

SLOPE_THRESHOLD = float(os.getenv("SLOPE_THRESHOLD", 30))
ELEVATION_LOW = float(os.getenv("ELEVATION_LOW", 50))
BUFFER_HIGH = float(os.getenv("BUFFER_HIGH", 500))
TARGET_COUNTY = os.getenv("TARGET_COUNTY", "花蓮縣")

# ---- Local paths (edit only if your files are elsewhere) ----
township_path = r"C:\Users\admin\Desktop\遙測\鄉(鎮、市、區)界線1140318\TOWN_MOI_1140318.shp"
shelter_csv_path = r"C:\Users\admin\Desktop\遙測\避難收容處所點位檔案v9.csv"
dem_path = r"C:\Users\admin\Desktop\遙測\dem_20m_hualien.tif"

# Water Resources Agency river polygon shapefile
WRA_URL = "https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP"

print({
    "TARGET_COUNTY": TARGET_COUNTY,
    "SLOPE_THRESHOLD": SLOPE_THRESHOLD,
    "ELEVATION_LOW": ELEVATION_LOW,
    "BUFFER_HIGH": BUFFER_HIGH,
})


## Captain's Log 2 — Load vector data

Load township boundaries, shelter CSV, and river polygons. Convert everything to **EPSG:3826** so distance and zonal statistics use meters.

In [ ]:

# Load township polygons
assert Path(township_path).exists(), f"Township shapefile not found: {township_path}"
townships = gpd.read_file(township_path).to_crs(epsg=3826)

# Target county polygons
county_towns = townships[townships["COUNTYNAME"] == TARGET_COUNTY].copy()
assert len(county_towns) > 0, f"No townships found for {TARGET_COUNTY}"
county_boundary = county_towns.dissolve().reset_index(drop=True)
county_boundary_buffer = county_boundary.copy()
county_boundary_buffer["geometry"] = county_boundary_buffer.buffer(1000)

# Load shelter CSV
assert Path(shelter_csv_path).exists(), f"Shelter CSV not found: {shelter_csv_path}"
shelter_df = pd.read_csv(shelter_csv_path)
shelter_df["經度"] = pd.to_numeric(shelter_df["經度"], errors="coerce")
shelter_df["緯度"] = pd.to_numeric(shelter_df["緯度"], errors="coerce")

# Filter target county shelters
county_shelter_df = shelter_df[
    shelter_df["縣市及鄉鎮市區"].astype(str).str.startswith(TARGET_COUNTY)
].dropna(subset=["經度", "緯度"]).copy()

shelters = gpd.GeoDataFrame(
    county_shelter_df,
    geometry=gpd.points_from_xy(county_shelter_df["經度"], county_shelter_df["緯度"]),
    crs="EPSG:4326"
).to_crs(epsg=3826)

# Load river polygons
rivers = gpd.read_file(WRA_URL).to_crs(epsg=3826)
rivers_in_county = gpd.sjoin(rivers, county_boundary, predicate="intersects")
print(f"河川面與目標縣市交集：{len(rivers_in_county)} 筆")
assert len(rivers_in_county) > 0, "⚠️ 河川資料未涵蓋目標縣市！請重新下載完整河川資料。"

# Limit rivers to target county extent for faster distance computation
rivers_county = gpd.overlay(rivers, county_boundary_buffer, how="intersection")

print("Townships:", len(townships))
print("County towns:", len(county_towns))
print("Shelters in county:", len(shelters))
print("Rivers in county:", len(rivers_county))
county_towns[["COUNTYNAME", "TOWNNAME", "geometry"]].head()


## Captain's Log 3 — River distance baseline

Compute each shelter's minimum distance to river polygons and convert the result into a distance category.

In [ ]:

# Minimum distance from each shelter point to any river polygon (meters)
shelters["river_distance_m"] = shelters.geometry.apply(lambda geom: rivers_county.distance(geom).min())

def river_distance_category(d):
    if d < 500:
        return "<500m"
    elif d < 1000:
        return "500-1000m"
    else:
        return ">=1000m"

shelters["river_distance_category"] = shelters["river_distance_m"].apply(river_distance_category)

shelters[["避難收容處所名稱", "river_distance_m", "river_distance_category"]].head()


## Captain's Log 4 — Load and clip DEM

Read the 20m DEM, verify metadata, then clip it to the target county boundary buffered by 1000 m. This keeps memory manageable while preserving coverage for 500 m shelter buffers.

In [ ]:

assert Path(dem_path).exists(), f"DEM not found: {dem_path}"
dem = rxr.open_rasterio(dem_path, masked=True)

print("DEM shape:", dem.shape)
print("DEM CRS:", dem.rio.crs)
print("DEM bounds:", dem.rio.bounds())
print("DEM resolution:", dem.rio.resolution())

clip_boundary = county_boundary_buffer.copy()
if dem.rio.crs != clip_boundary.crs:
    clip_boundary = clip_boundary.to_crs(dem.rio.crs)

dem_clip = dem.rio.clip(clip_boundary.geometry, clip_boundary.crs, drop=True)

print("Clipped DEM shape:", dem_clip.shape)
print("Clipped DEM bounds:", dem_clip.rio.bounds())


## Captain's Log 5 — Slope from DEM

Convert the clipped DEM to a NumPy array, respect NoData, and compute slope in degrees using the DEM's 20 m pixel spacing.

In [ ]:

dem_arr = dem_clip.values[0].astype("float64")

# Handle nodata / masked pixels
if np.ma.isMaskedArray(dem_arr):
    dem_arr = dem_arr.filled(np.nan)

res_x, res_y = dem_clip.rio.resolution()
pixel_size = abs(res_x)  # expected 20 m

dy, dx = np.gradient(dem_arr, pixel_size)
slope_arr = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

dem_affine = dem_clip.rio.transform()

print("Pixel size:", pixel_size)
print("Elevation min/max:", np.nanmin(dem_arr), np.nanmax(dem_arr))
print("Slope min/max:", np.nanmin(slope_arr), np.nanmax(slope_arr))


## Captain's Log 6 — Zonal statistics for 500 m shelter buffers

Generate 500 m buffers around shelters and extract mean elevation, elevation standard deviation, and maximum slope for each buffer.

In [ ]:

shelter_buffers = shelters.copy()
shelter_buffers["geometry"] = shelter_buffers.buffer(BUFFER_HIGH)

# Elevation stats
zs_elev = zonal_stats(
    shelter_buffers,
    dem_arr,
    affine=dem_affine,
    stats=["mean", "std"],
    nodata=np.nan,
)

# Slope stats
zs_slope = zonal_stats(
    shelter_buffers,
    slope_arr,
    affine=dem_affine,
    stats=["max"],
    nodata=np.nan,
)

terrain_df = pd.DataFrame(zs_elev).rename(columns={
    "mean": "mean_elevation",
    "std": "std_elevation",
})
terrain_df["max_slope"] = pd.DataFrame(zs_slope)["max"]

shelters = shelters.reset_index(drop=True)
shelters = pd.concat([shelters, terrain_df], axis=1)

shelters[[
    "避難收容處所名稱",
    "river_distance_m",
    "mean_elevation",
    "std_elevation",
    "max_slope"
]].head()


## Captain's Log 7 — Composite risk logic

Apply the homework rules:
- **very_high**: river distance < 500 m and max slope > threshold
- **high**: river distance < 500 m or max slope > threshold
- **medium**: river distance < 1000 m and mean elevation < threshold
- **low**: otherwise

In [ ]:

def classify_risk(row):
    d = row["river_distance_m"]
    s = row["max_slope"]
    e = row["mean_elevation"]

    if pd.notna(d) and pd.notna(s) and d < BUFFER_HIGH and s > SLOPE_THRESHOLD:
        return "very_high"
    elif (pd.notna(d) and d < BUFFER_HIGH) or (pd.notna(s) and s > SLOPE_THRESHOLD):
        return "high"
    elif pd.notna(d) and pd.notna(e) and d < 1000 and e < ELEVATION_LOW:
        return "medium"
    else:
        return "low"

shelters["risk_level"] = shelters.apply(classify_risk, axis=1)
shelters["risk_level"].value_counts()


## Captain's Log 8 — Visualization

Build a DEM-based risk map and a top-10 scatter plot. The map uses DEM as background and colors shelters by composite risk.

In [ ]:

# Simple hillshade from slope/aspect approximation
azimuth = np.radians(315)
altitude = np.radians(45)

grad_y, grad_x = np.gradient(dem_arr, pixel_size)
slope_rad = np.arctan(np.sqrt(grad_x**2 + grad_y**2))
aspect_rad = np.arctan2(-grad_x, grad_y)
hillshade = (
    np.sin(altitude) * np.cos(slope_rad) +
    np.cos(altitude) * np.sin(slope_rad) * np.cos(azimuth - aspect_rad)
)
hillshade = np.clip(hillshade, 0, 1)

xmin, ymin, xmax, ymax = dem_clip.rio.bounds()
risk_colors = {"very_high": "red", "high": "orange", "medium": "gold", "low": "green"}

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(hillshade, cmap="gray", extent=(xmin, xmax, ymin, ymax), origin="upper")
county_boundary.boundary.plot(ax=ax, color="black", linewidth=1)

for level, color in risk_colors.items():
    subset = shelters[shelters["risk_level"] == level]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, markersize=12, label=level, alpha=0.9)

ax.set_title(f"{TARGET_COUNTY} DEM Hillshade + Shelter Terrain Risk")
ax.legend(title="risk_level")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
plt.tight_layout()
plt.savefig("terrain_risk_map.png", dpi=300, bbox_inches="tight")
plt.show()

# Top 10 high-risk scatter plot
risk_rank = pd.Categorical(shelters["risk_level"], categories=["very_high", "high", "medium", "low"], ordered=True)
top10 = shelters.assign(_rank=risk_rank).sort_values(["_rank", "max_slope", "mean_elevation"], ascending=[True, False, True]).head(10)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(top10["mean_elevation"], top10["max_slope"])
for _, row in top10.iterrows():
    ax.annotate(row["避難收容處所名稱"], (row["mean_elevation"], row["max_slope"]), fontsize=8)
ax.set_xlabel("Mean elevation (m)")
ax.set_ylabel("Max slope (deg)")
ax.set_title("Top 10 Higher-Risk Shelters: Slope vs Elevation")
plt.tight_layout()
plt.show()


## Captain's Log 9 — Save deliverables

Export the required homework deliverables:
- `terrain_risk_audit.json`
- `terrain_risk.geojson`
- `terrain_risk_map.png` (already saved in the previous cell)

In [ ]:

# Prepare output columns required by the homework
shelters_final = shelters.copy().reset_index(drop=True)
shelters_final.insert(0, "shelter_id", shelters_final.index + 1)

# JSON audit table (no geometry)
audit_df = shelters_final[[
    "shelter_id",
    "避難收容處所名稱",
    "risk_level",
    "mean_elevation",
    "max_slope",
    "river_distance_category",
]].rename(columns={"避難收容處所名稱": "name"})

audit_df.to_json(
    "terrain_risk_audit.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

# GeoJSON with safe, short field names
export_gdf = shelters_final.rename(columns={
    "避難收容處所名稱": "name",
    "mean_elevation": "mean_elev",
    "std_elevation": "std_elev",
    "max_slope": "max_slope",
    "river_distance_m": "riv_dist_m",
    "river_distance_category": "riv_dist_c",
    "risk_level": "risk_lvl",
})[[
    "shelter_id",
    "name",
    "mean_elev",
    "std_elev",
    "max_slope",
    "riv_dist_m",
    "riv_dist_c",
    "risk_lvl",
    "geometry",
]].copy()

export_gdf.to_file("terrain_risk.geojson", driver="GeoJSON")

print("Saved terrain_risk_audit.json")
print("Saved terrain_risk.geojson")
print("Saved terrain_risk_map.png")
audit_df.head()
